# Image-Based Animal Action Recognition with ResNet-50

**Objective:** Predict an animal action from a single RGB image.

| Component | Specification |
| :--- | :--- |
| Classes | Keeping still, Walking, Eating |
| Input | One RGB image, transformed to a 3 × 224 × 224 tensor |
| Backbone | ResNet-50 with ImageNet-1K V2 pretrained weights |
| Trainable component | A linear classifier over 2,048-dimensional features |
| Output | Three class scores; the highest-probability action is returned |
| Evaluation | Accuracy, Macro-F1, and a confusion matrix |

Select a Python kernel with the dependencies listed in `requirements.txt` and run the cells in order. If the supplied kernel is installed, select **Python (AnimalAct)**. The first run downloads approximately 98 MB of pretrained weights; subsequent runs use the local cache. Apple MPS or CUDA accelerates feature extraction when available; CPU execution is supported.

This is a single-image classification exercise. Annotated videos supply labeled images during dataset preparation. At inference time, the model receives only the image pixels. Temporal video grounding remains a final-project extension.

## 1. Data Loading and Preprocessing

### 1.1 Environment and Experimental Configuration

This introductory exercise uses three classes from the unified main-evaluation list. It reads the released train/valid/test assignments directly. Hyperparameters are selected on valid, and the final head is fitted on train only. There is no random re-splitting or train+valid refit. This three-class image task does not measure performance on all main-evaluation classes.


In [ ]:
import hashlib
import importlib.metadata
import json
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights

SEED = 42
CLASS_NAMES = ["Keeping still", "Walking", "Eating"]
MIN_SPAN_SECONDS = 0.5
BATCH_SIZE = 16
C_VALUES = [0.01, 0.1, 1.0, 10.0]
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(4)
cv2.setNumThreads(1)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu" and torch.backends.mps.is_available():
    DEVICE = "mps"
started = time.perf_counter()
print("Device:", DEVICE)

In [ ]:
DATASET_DIR = None
for location in [Path(".."), Path("."), Path("new_data/AnimalAct")]:
    if ((location / "manifest.json").is_file()
            and (location / "annotations").is_dir()
            and (location / "videos").is_dir()):
        DATASET_DIR = location.resolve()
        break
if DATASET_DIR is None:
    raise FileNotFoundError("Launch from the dataset demo folder, the dataset root, or the repository root.")
DEMO_DIR = DATASET_DIR / "demo"
IMAGE_DIR = DEMO_DIR / "images"
OUTPUT_DIR = DEMO_DIR / "outputs_resnet50_fixed_v1"
IMAGE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
manifest_path = DATASET_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
samples = manifest["samples"]
assert len(samples) == manifest["count"]
assert len({row["video_id"] for row in samples}) == len(samples)
assert all(row["split"] in {"train", "valid", "test"} for row in samples)
print("Source videos:", len(samples))
split_version = manifest["split_version"]
main_evaluation = json.loads((DATASET_DIR / "main_evaluation_classes.json").read_text())
vocabulary = json.loads((DATASET_DIR / "action_classes.json").read_text())
name_to_global_id = {c["name"]: c["class_id"] for c in vocabulary["classes"]}
assert set(CLASS_NAMES) <= set(main_evaluation["class_names"])
GLOBAL_CLASS_IDS = [name_to_global_id[name] for name in CLASS_NAMES]
print("Fixed split:", split_version)
print("Unified main-evaluation classes:", main_evaluation["num_classes"])
print("Demo local index -> global class ID:", GLOBAL_CLASS_IDS)


### 1.2 Image–Label Association

Each manifest entry links `video_id`, `video_path`, and `annotation_path`. The annotation supplies action names and time intervals. We retain spans with one distinct active action across all animals, merging adjacent spans with the same label. Other overlapping actions exclude a span even when they are outside our class vocabulary.

For each video and target action, take the midpoint frame of the longest eligible span of at least 0.5 seconds. Save that frame as a PNG and associate its path with a numeric target. The local model-output mapping is **0 = Keeping still, 1 = Walking, 2 = Eating**. These indices follow `CLASS_NAMES`; the dataset-wide class IDs are documented in [ACTION_CLASSES.md](../ACTION_CLASSES.md). A video can supply images for different actions, but all of its images stay in the same split. Only the PNG pixels enter the model; video IDs and labels are metadata, not input features.

This policy inherits the video interval labels. A single frame may not contain enough motion information to distinguish all actions. Source videos and annotations are not modified.

In [ ]:
def unambiguous_spans(annotation, duration):
    """Return (start, end, action) spans with one distinct active action."""
    intervals = []
    for animal in annotation["annotations"]:
        for item in animal["actions"]:
            start = max(0.0, float(item["start"]))
            end = min(duration, float(item["end"]))
            if start < end:
                intervals.append((start, end, item["action"]))

    boundaries = sorted({t for start, end, _ in intervals for t in (start, end)})
    spans = []
    for start, end in zip(boundaries, boundaries[1:]):
        midpoint = (start + end) / 2
        active = {label for left, right, label in intervals if left <= midpoint < right}
        if len(active) != 1:
            continue
        label = next(iter(active))
        same_action = bool(spans) and spans[-1][2] == label
        adjacent = bool(spans) and abs(spans[-1][1] - start) < 1e-8
        if same_action and adjacent:
            spans[-1] = (spans[-1][0], end, label)
        else:
            spans.append((start, end, label))
    return spans

In [ ]:
def make_image_records(samples, dataset_dir, image_dir):
    records = []
    for sample in samples:
        annotation_path = dataset_dir / sample["annotation_path"]
        annotation = json.loads(annotation_path.read_text(encoding="utf-8"))
        capture = cv2.VideoCapture(str(dataset_dir / sample["video_path"]))
        try:
            fps = capture.get(cv2.CAP_PROP_FPS)
            count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
            if not capture.isOpened() or not np.isfinite(fps) or fps <= 0 or count < 1:
                raise RuntimeError(f"Invalid video metadata: {sample['video_id']}")
            spans = unambiguous_spans(annotation, count / fps)
            for target, label in enumerate(CLASS_NAMES):
                eligible = [(s, e) for s, e, a in spans
                            if a == label and e - s >= MIN_SPAN_SECONDS]
                if not eligible:
                    continue
                start, end = max(eligible, key=lambda span: span[1] - span[0])
                frame_index = min(count - 1, int((start + end) * fps / 2))
                capture.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
                ok, bgr = capture.read()
                if not ok:
                    raise RuntimeError(f"Cannot decode {sample['video_id']}")
                filename = f"{sample['video_id']}_{frame_index:06d}.png"
                if not cv2.imwrite(str(image_dir / filename), bgr):
                    raise OSError(f"Cannot write image: {filename}")
                records.append(dict(
                    video_id=sample["video_id"], source_split=sample["split"],
                    split_group_id=sample["split_group_id"],
                    image_path=f"images/{filename}", frame_index=frame_index,
                    timestamp=frame_index / fps, label=label, target=target,
                ))
        finally:
            capture.release()
    return records

### 1.3 Fixed Splits and Duplicate-Image Audit

All images inherit their video's released split. No videos are silently removed or reassigned. Check exact selected PNG hashes across all three sets; abort if a cross-split collision is found so the dataset release can be reviewed. Within-split duplicates remain and are counted. Exact hashes cannot rule out near-duplicates.


In [ ]:
records = make_image_records(samples, DATASET_DIR, IMAGE_DIR)
hash_splits = {}
for row in records:
    row["image_sha256"] = hashlib.sha256((DEMO_DIR / row["image_path"]).read_bytes()).hexdigest()
    hash_splits.setdefault(row["image_sha256"], set()).add(row["source_split"])
duplicate_cross_split_hashes = sorted(h for h, splits in hash_splits.items() if len(splits) > 1)
if duplicate_cross_split_hashes:
    raise RuntimeError("Exact images cross fixed splits; review the release instead of re-splitting locally.")
print("Cross-split exact image duplicates:", len(duplicate_cross_split_hashes))


In [ ]:
targets = np.array([row["target"] for row in records])
groups = np.array([row["split_group_id"] for row in records])
train_indices = np.array([i for i, row in enumerate(records)
                           if row["source_split"] == "train"])
test_indices = np.array([i for i, row in enumerate(records)
                         if row["source_split"] == "test"])
fit_indices = train_indices
validation_indices = np.array([i for i, row in enumerate(records)
                               if row["source_split"] == "valid"], dtype=int)
partitions = {"Fit": fit_indices, "Validation": validation_indices, "Test": test_indices}
for left, right in [("Fit", "Validation"), ("Fit", "Test"), ("Validation", "Test")]:
    assert set(groups[partitions[left]]).isdisjoint(groups[partitions[right]])
for indices in partitions.values():
    assert set(targets[indices]) == set(range(len(CLASS_NAMES)))
summary = ["| Action | Fit | Validation | Test |", "| :--- | ---: | ---: | ---: |"]
for target, label in enumerate(CLASS_NAMES):
    counts = [int((targets[indices] == target).sum()) for indices in partitions.values()]
    summary.append(f"| {label} | {counts[0]} | {counts[1]} | {counts[2]} |")
display(Markdown("\n".join(summary)))
print("Images:", len(records))

### 1.4 Image Dataset and Input Tensor

The pretrained weights define the preprocessing: resize the shorter edge to 232 pixels, center-crop to 224 × 224, convert to a tensor, and normalize using ImageNet channel statistics.

| Stage | Shape |
| :--- | :--- |
| Source RGB image | H × W × 3 |
| Transformed image | 3 × 224 × 224 |
| Input batch | N × 3 × 224 × 224 |
| Backbone features | N × 2048 |
| Model output | N × 3 |

The original frame is kept as a PNG. Normalization occurs when the image is loaded, not when it is saved.

In [ ]:
weights = ResNet50_Weights.IMAGENET1K_V2
preprocess = weights.transforms()


class ActionImageDataset(Dataset):
    def __init__(self, records, root, transform):
        self.records = records
        self.root = root
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        with Image.open(self.root / row["image_path"]) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, row["target"]


image_dataset = ActionImageDataset(records, DEMO_DIR, preprocess)
image_loader = DataLoader(
    image_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
)
example_image, example_target = image_dataset[0]
assert example_image.shape == (3, 224, 224)
print("One image:", tuple(example_image.shape))
print("Its label:", CLASS_NAMES[example_target])

## 2. Model Training

### 2.1 Pretrained ResNet-50 Backbone

The architecture is:

```text
RGB image [N, 3, 224, 224]
    → Convolution stem and max pooling
    → Four residual stages with 3, 4, 6, and 3 bottleneck blocks
    → Global average pooling
    → Feature vector [N, 2048]
    → Linear classifier [2048 → 3]
    → Class logits [N, 3]
```

The original 1,000-class ImageNet head is removed. We freeze the convolutional backbone and keep it in evaluation mode, which also fixes its batch-normalization statistics. Features can then be extracted once and reused while fitting the small classification head. No averaging across video frames is performed.

In [ ]:
model = resnet50(weights=weights)
model.fc = nn.Identity()
model.requires_grad_(False)
model.eval().to(DEVICE)

feature_batches = []
feature_start = time.perf_counter()
with torch.inference_mode():
    for images, _ in image_loader:
        features = model(images.to(DEVICE))
        feature_batches.append(features.cpu().numpy())
X = np.concatenate(feature_batches)
assert X.shape == (len(records), 2048)
assert np.isfinite(X).all()
feature_seconds = time.perf_counter() - feature_start
print("Feature matrix:", X.shape)
print(f"Feature extraction: {feature_seconds:.1f} seconds")

### 2.2 Classification Head and Validation

Multiclass logistic regression fits a linear head with three outputs. Its softmax probabilities are equivalent to applying a PyTorch linear layer followed by softmax. We use scikit-learn's optimizer to keep this demonstration short and fast; only the head is learned.

Inverse-frequency class weights are computed from the fitting labels. Choose the regularization parameter `C` using validation Accuracy, breaking ties with Macro-F1. After selection, refit the head on the fixed train split. The test split is not used for either decision.

In [ ]:
validation_results = []
for c_value in C_VALUES:
    candidate = LogisticRegression(
        C=c_value, max_iter=1000, class_weight="balanced", random_state=SEED,
    )
    candidate.fit(X[fit_indices], targets[fit_indices])
    predictions = candidate.predict(X[validation_indices])
    validation_results.append(dict(
        C=c_value,
        accuracy=float(accuracy_score(targets[validation_indices], predictions)),
        macro_f1=float(f1_score(
            targets[validation_indices], predictions, average="macro", zero_division=0,
        )),
    ))
best = max(validation_results, key=lambda row: (row["accuracy"], row["macro_f1"]))
comparison = ["| C | Validation Accuracy | Validation Macro-F1 |",
              "| ---: | ---: | ---: |"]
for row in validation_results:
    comparison.append(f"| {row['C']} | {row['accuracy']:.3f} | {row['macro_f1']:.3f} |")
display(Markdown("\n".join(comparison)))
print("Selected C:", best["C"])

classifier = LogisticRegression(
    C=best["C"], max_iter=1000, class_weight="balanced", random_state=SEED,
)
classifier.fit(X[train_indices], targets[train_indices])

### 2.3 Complete Image-to-Action Model

Copy the fitted linear weights into `model.fc`. The resulting ResNet-50 accepts an image batch directly and outputs three logits. Label order is checked before the copy. The equivalence check below uses training examples, not test data.

In [ ]:
assert np.array_equal(classifier.classes_, np.arange(len(CLASS_NAMES)))
model.fc = nn.Linear(2048, len(CLASS_NAMES))
with torch.no_grad():
    model.fc.weight.copy_(torch.from_numpy(classifier.coef_).float())
    model.fc.bias.copy_(torch.from_numpy(classifier.intercept_).float())
model.eval().to(DEVICE)
with torch.inference_mode():
    feature_tensor = torch.from_numpy(X[train_indices[:8]]).to(DEVICE)
    torch_probabilities = model.fc(feature_tensor).softmax(dim=1).cpu().numpy()
np.testing.assert_allclose(
    torch_probabilities, classifier.predict_proba(X[train_indices[:8]]),
    atol=1e-5, rtol=1e-4,
)
print("Backbone parameters: frozen")
print("Fitted head:", model.fc)
print("Head parameters:", sum(parameter.numel() for parameter in model.fc.parameters()))

## 3. Model Evaluation

### 3.1 Held-Out Image Classification

Evaluate every eligible test image. Accuracy and Macro-F1 are image-level metrics. Images originating from the same video may be correlated, so this is not a collection of fully independent observations.

The test set retains its observed class distribution. A majority-class reference always predicts the most frequent fixed train label, providing a simple reference for interpreting Accuracy.

In [ ]:
test_rows = [records[int(index)] for index in test_indices]
test_loader = DataLoader(
    ActionImageDataset(test_rows, DEMO_DIR, preprocess),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
)
probability_batches = []
with torch.inference_mode():
    for images, _ in test_loader:
        logits = model(images.to(DEVICE))
        probability_batches.append(logits.softmax(dim=1).cpu().numpy())
probabilities = np.concatenate(probability_batches)
y_test = targets[test_indices]
y_pred = probabilities.argmax(axis=1)
accuracy = float(accuracy_score(y_test, y_pred))
macro_f1 = float(f1_score(y_test, y_pred, average="macro", zero_division=0))
majority_label = np.bincount(targets[train_indices]).argmax()
majority_accuracy = float((y_test == majority_label).mean())
display(Markdown(
    "| Model | Test Accuracy | Test Macro-F1 |\n| :--- | ---: | ---: |\n"
    f"| ResNet-50 + linear head | {accuracy:.3f} | {macro_f1:.3f} |\n"
    f"| Majority-class reference | {majority_accuracy:.3f} | — |"
))
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3))


### 3.2 Single-Image Inference

`predict_image` accepts an image path and returns an action label and its softmax probability. It does not read an annotation, a video ID, or a timestamp. The confidence value is a model probability, not a guarantee of correctness.

To try another RGB image, change `image_path` in the next cell. The model can predict only the three supported actions; it does not include an unknown-action detector.

In [ ]:
def predict_image(image_path, model, transform, device, class_names):
    with Image.open(image_path) as image:
        tensor = transform(image.convert("RGB")).unsqueeze(0).to(device)
    model.eval()
    with torch.inference_mode():
        probabilities = model(tensor).softmax(dim=1)[0].cpu()
    index = int(probabilities.argmax())
    return {"action": class_names[index], "confidence": float(probabilities[index])}


image_path = DEMO_DIR / test_rows[0]["image_path"]
prediction = predict_image(image_path, model, preprocess, DEVICE, CLASS_NAMES)
print(json.dumps(prediction, indent=2))

## 4. Prediction Visualization and Error Analysis

### 4.1 Confusion Matrix

Rows show true labels and columns show predicted labels. Counts reveal whether performance differs across the three actions.

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 5.2))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=np.arange(len(CLASS_NAMES)),
    display_labels=CLASS_NAMES, cmap="Blues", colorbar=False,
    values_format="d", ax=ax,
)
ax.set_title("ResNet-50: held-out image classification")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=160)
plt.show()
plt.close(fig)

### 4.2 Image Predictions and Errors

For each class, show the first correct and first incorrect prediction when available. Empty panels indicate that no example of that type exists. This fixed selection rule avoids displaying only favorable predictions. The figure shows original images; the network receives their normalized center crops.

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 2, figsize=(9, 10), squeeze=False)
for target, label in enumerate(CLASS_NAMES):
    indices = np.flatnonzero(y_test == target)
    for column, correct in enumerate([True, False]):
        eligible = indices[(y_pred[indices] == target) == correct]
        ax = axes[target, column]
        ax.axis("off")
        if not len(eligible):
            ax.set_title(f"{label}: no {'correct' if correct else 'incorrect'} example")
            continue
        index = int(eligible[0])
        with Image.open(DEMO_DIR / test_rows[index]["image_path"]) as image:
            ax.imshow(image.convert("RGB"))
        confidence = probabilities[index, y_pred[index]]
        ax.set_title(
            f"True: {label}\nPredicted: {CLASS_NAMES[y_pred[index]]} ({confidence:.1%})",
            fontsize=10,
        )
fig.suptitle("Single-image predictions", fontsize=14)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "prediction_examples.png", dpi=160)
plt.show()
plt.close(fig)

### 4.3 Saved Model and Experiment Record

Save the complete PyTorch model state, class order, preprocessing configuration, source-image mapping, fixed split version, and metrics. `resnet50_action.pt` contains the fitted weights, so deployment does not require the source annotations or the scikit-learn classifier.

To restore the model, construct `resnet50(weights=None)`, replace its head with `nn.Linear(2048, 3)`, and load the checkpoint's `state_dict` with `torch.load(..., weights_only=True)`. Apply the same preprocessing and class order used here.

In [ ]:
checkpoint = {
    "state_dict": {name: value.detach().cpu() for name, value in model.state_dict().items()},
    "class_names": CLASS_NAMES,
    "global_class_ids": GLOBAL_CLASS_IDS, "split_version": split_version,
    "pretrained_weights": weights.name,
    "input_shape": [3, 224, 224],
    "resize_short_edge": 232,
    "mean": [0.485, 0.456, 0.406],
    "std": [0.229, 0.224, 0.225],
}
torch.save(checkpoint, OUTPUT_DIR / "resnet50_action.pt")
report = {
    "task": "single-image action classification", "seed": SEED,
    "split_version": split_version, "global_class_ids": GLOBAL_CLASS_IDS,
    "training_policy": "train only; select hyperparameters on valid",
    "classes": CLASS_NAMES, "selected_C": best["C"],
    "manifest_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    "duplicate_cross_split_hashes": sorted(duplicate_cross_split_hashes),
    "fit_images": len(fit_indices), "validation_images": len(validation_indices),
    "final_training_images": len(train_indices), "test_images": len(test_indices),
    "validation_results": validation_results,
    "accuracy": accuracy, "macro_f1": macro_f1, "majority_accuracy": majority_accuracy,
    "feature_seconds": feature_seconds, "total_seconds": time.perf_counter() - started,
    "fit_indices": fit_indices.tolist(), "validation_indices": validation_indices.tolist(),
    "test_indices": test_indices.tolist(), "records": records,
    "predictions": y_pred.tolist(), "probabilities": probabilities.tolist(),
    "versions": {name: importlib.metadata.version(name) for name in
                 ["torch", "torchvision", "numpy", "scikit-learn", "opencv-python"]},
}
(OUTPUT_DIR / "results.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved model and report:", OUTPUT_DIR.relative_to(DATASET_DIR))
print(f"Total elapsed time: {report['total_seconds']:.1f} seconds")

## Extensions and Final Project Objectives

The current model predicts an action from one image. Some actions depend on motion, so a still image may be ambiguous even when its source interval has a clear label. For the final action-recognition task, compare image-based features with models that use temporal information and report Accuracy, Macro-F1, and a Confusion Matrix under the same evaluation protocol.

For optional temporal video grounding, accept an untrimmed video and a natural-language query, then predict the matching time interval. Report temporal IoU, mIoU, and R@1 at IoU thresholds 0.5 and 0.7. State the reference-matching policy when a query has multiple valid intervals.

### Documentation

- [Torchvision pretrained models](https://docs.pytorch.org/vision/stable/models.html)
- [PyTorch transfer learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)